In [1]:
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras import layers, Model
from sklearn.model_selection import train_test_split
from datasets import load_dataset

c:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
dataset = load_dataset("ucirvine/sms_spam")
texts = np.array(dataset["train"]["sms"])
labels = np.array(dataset["train"]["label"])
X_train, X_val, y_train, y_val = train_test_split(texts, labels, test_size=0.2, random_state=42, stratify=labels)

c:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lenovo\.cache\huggingface\hub\datasets--ucirvine--sms_spam. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating train split: 100%|██████████| 5574/5574 [00:00<00:00, 9638.53 examples/s]


In [6]:
input_layer = Input(shape=(), dtype=tf.string, name="text_input")
preprocessor_layer = hub.KerasLayer("https://www.kaggle.com/models/tensorflow/bert/TensorFlow2/en-uncased-preprocess/3", name="preprocessing")
encoder_layer = hub.KerasLayer("https://www.kaggle.com/models/tensorflow/bert/TensorFlow2/bert-en-uncased-l-12-h-128-a-2/2", trainable=True, name="BERT_encoder")

RuntimeError: Op type not registered 'CaseFoldUTF8' in binary running on DESKTOP-9HAUF76. Make sure the Op and Kernel are registered in the binary running in this process. Note that if you are loading a saved graph which used ops from tf.contrib (e.g. `tf.contrib.resampler`), accessing should be done before importing the graph, as contrib ops are lazily registered when the module is first accessed.

In [ ]:
x = preprocessor_layer(input_layer)
x = encoder_layer(x)["pooled_output"]     
x = Dropout(0.1, name="dropout")(x)
output_layer = Dense(2, activation="softmax", name="output")(x)  # spam or not spam
model = Model(inputs=input_layer, outputs=output_layer)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

In [ ]:
model.summary()

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=5,
    batch_size=32
)

In [ ]:
preds = model.predict(["Congratulations! You won a free prize, click here"])
print(preds)